# ML Baseline: Predict next-day extreme loss (SPY)

**Updated:** 2026-03-10  
**Goal:** Build a minimal time-series ML pipeline (no leakage) and diagnose why a baseline may fail on rare events.

We define the label using a train-only quantile threshold on next-day loss:
- `loss_t = -ret_t`
- `y_t = 1{ loss_{t+1} > q(label_q) }`

## 1) Setup (imports)

We use:
- `pandas/numpy` for time-series features
- `scikit-learn` for a baseline classifier (Logistic Regression)

In [8]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

## 2) Data + feature engineering (price → returns → loss)

**Input:** daily adjusted price series (`date`, `price`)  
We compute:
- log return: $r_t = \log(P_t) - \log(P_{t-1}) $
- loss: $ L_t = -r_t $

**Features at time $t$:**
- `ret_t`: today’s log return
- `vol20_t`: rolling 20-day volatility of returns
- `var250_t`: rolling 250-day 1% quantile of returns (left-tail risk)

**Label (target):**  
$ y_t = 1\{ L_{t+1} > q_{\text{label\_q}}(L_{t+1}) \} $


Important: the quantile threshold is computed on train only to avoid leakage.

In [9]:
label_q = 0.90  # we tried 0.95 and 0.90 to avoid "no positives" in test

df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)

df_feat = pd.DataFrame(index=df.index)
df_feat["ret"] = np.log(price).diff()
df_feat["loss"] = -df_feat["ret"]

df_feat["vol20"] = df_feat["ret"].rolling(20).std()
df_feat["var250"] = df_feat["ret"].rolling(250).quantile(0.01)

df_feat["loss_t1"] = df_feat["loss"].shift(-1)

feat_cols = ["ret", "vol20", "var250"]
df_feat = df_feat.dropna(subset=feat_cols + ["loss_t1"]).copy()



## 3) Time-based split + leakage-safe label

**Split:** chronological 80/20 split (no shuffle).  
Shuffling time series can leak future information into training.

**Label definition (predicting tomorrow):**
- We predict whether tomorrow is an extreme-loss day.
- Define `loss_t = -ret_t` and `loss_t1 = loss_{t+1}` via shifting.
- Compute the extreme-loss cutoff on the training set only:
  - `thr = quantile(train.loss_t1, label_q)`
- Then define labels:
  - `y_t = 1{ loss_t1 > thr }`

We print class balance (positives in train/test) to understand rarity.

In [10]:
split = int(len(df_feat) * 0.8)
train = df_feat.iloc[:split]
test = df_feat.iloc[split:]

# train-only threshold (avoid leakage)
thr = train["loss_t1"].quantile(label_q)

X_train = train[feat_cols].to_numpy()
X_test = test[feat_cols].to_numpy()

y_train = (train["loss_t1"] > thr).astype(int).to_numpy()
y_test = (test["loss_t1"] > thr).astype(int).to_numpy()

print(f"thr (train-only, q={label_q}) =", float(thr))
print("train positives:", int(y_train.sum()), "/", len(y_train), f"({y_train.mean():.4f})")
print("test positives :", int(y_test.sum()), "/", len(y_test), f"({y_test.mean():.4f})")

thr (train-only, q=0.9) = 0.012771045393253737
train positives: 81 / 804 (0.1007)
test positives : 8 / 201 (0.0398)


## 4) Model + probability diagnostics

**Model:** Logistic Regression baseline with `StandardScaler`
- scaling stabilizes optimization and makes coefficients comparable across features
- this baseline checks whether simple signals exist in our features

**Diagnostics:**
Before choosing a classification threshold, we inspect predicted probabilities:
- distribution summary (min/median/max)
- mean probability for `y=0` vs `y=1`
- ROC-AUC (only meaningful if both classes exist in test)

If probabilities overlap heavily or `mean proba(y=1) ≤ mean proba(y=0)`,
the baseline is not separating the rare event well.

In [11]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, solver="lbfgs")  # (we also tested class_weight="balanced")
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

print("\n=== proba summary ===")
print(pd.Series(proba).describe())

print("\n=== mean proba by class ===")
proba_s = pd.Series(proba, index=test.index)
y_s = pd.Series(y_test, index=test.index)
print("mean proba (y=0):", float(proba_s[y_s==0].mean()))
print("mean proba (y=1):", float(proba_s[y_s==1].mean()))

if len(np.unique(y_test)) == 2:
    print("ROC-AUC:", roc_auc_score(y_test, proba))
else:
    print("ROC-AUC: not defined (only one class in y_test)")


=== proba summary ===
count    201.000000
mean       0.087284
std        0.007812
min        0.073178
25%        0.081719
50%        0.086192
75%        0.092805
max        0.113025
dtype: float64

=== mean proba by class ===
mean proba (y=0): 0.08736911191309613
mean proba (y=1): 0.08522634844706838
ROC-AUC: 0.4514248704663213


## 5) Threshold sweep (precision–recall trade-off)

Rare-event detection depends heavily on the decision threshold.

For each threshold $t$, we compute:
- `TP, FP, FN, TN`
- precision $= \frac{TP}{TP+FP}$
- recall $= \frac{TP}{TP+FN}$

Interpretation:
- Higher recall catches more extreme-loss days but may increase false alarms (FP).
- Precision–recall trade-off is expected in risk alerting systems.

In [12]:
thresholds = [0.5, 0.3, 0.2, 0.1, 0.05]

print("\n=== Threshold sweep on test ===")
print("thr  pred_pos  TP  FP  FN  TN  precision  recall")

for t in thresholds:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pred_pos = int(pred.sum())

    print(f"{t:>4.2f} {pred_pos:>8d} {tp:>3d} {fp:>3d} {fn:>3d} {tn:>3d} "
          f"{precision:>9.4f} {recall:>7.4f}")


=== Threshold sweep on test ===
thr  pred_pos  TP  FP  FN  TN  precision  recall
0.50        0   0   0   8 193    0.0000  0.0000
0.30        0   0   0   8 193    0.0000  0.0000
0.20        0   0   0   8 193    0.0000  0.0000
0.10       13   0  13   8 180    0.0000  0.0000
0.05      201   8 193   0   0    0.0398  1.0000


## 6) Summary
- Time-series split (no shuffle) + train-only threshold prevents leakage.
- Rare-event classification is highly sensitive to thresholds and class imbalance.
- With simple features (ret/vol20/var250) + logistic regression, predicted probabilities were narrowly distributed and did not separate y=1 vs y=0 well in this run.

---

# ML Baseline: Predict next-day extreme loss (SPY) using lagged returns

**Updated:** 2026-03-16  
**Goal:** Build a simple time-series classification baseline to predict **next-day extreme loss** using
- lagged returns (`ret_lag1`…`ret_lag5`)
- volatility state (`vol20`)
- tail state (`var250`)

Evaluate with **rare-event friendly** metrics:
- ROC-AUC (ranking quality)
- Alert budget metrics: **Recall@K** (top-K alerts)

**Key idea:**
Features use information available at time $t$ only, while the label is defined using $L_{t+1}$ (next day).

## 1) Load data + Build features/label 

**Feature/label definition (leakage-safe):**

- Log return and loss:
  - $r_t = \log(P_t) - \log(P_{t-1})$
  - $L_t = -r_t$

- Next-day label:
  - $y_t = 1\{L_{t+1} > \mathrm{thr}\}$

- Train-only threshold:
  - $\mathrm{thr}$ is computed **only on train** using the label quantile $\mathrm{label\_q}$.
  - Test labels use the same fixed $\mathrm{thr}$.

In [21]:
import numpy as np
import pandas as pd

# --- knobs ---
label_q = 0.90   # top 10% tomorrow-loss days are positives
max_lag = 5

# 1) Load SPY prices
df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)

# 2) Build returns / loss
feat = pd.DataFrame(index=df.index)
feat["ret"] = np.log(price).diff()
feat["loss"] = -feat["ret"]

# 3) State features (using info up to time t)
feat["vol20"] = feat["ret"].rolling(20).std()
feat["var250"] = feat["ret"].rolling(250).quantile(0.01)

# 4) Lagged return features (past-only)
for k in range(1, max_lag + 1):
    feat[f"ret_lag{k}"] = feat["ret"].shift(k)

# 5) Label target = next-day loss
feat["loss_t1"] = feat["loss"].shift(-1)

feat_cols = ["ret", "vol20", "var250"] + [f"ret_lag{k}" for k in range(1, max_lag + 1)]

# Drop NaNs from diff/rolling/shift
feat = feat.dropna(subset=feat_cols + ["loss_t1"]).copy()

# 6) Time-based split (80/20)
split = int(len(feat) * 0.8)
train = feat.iloc[:split]
test = feat.iloc[split:]

# Train-only threshold (avoid leakage)
thr = train["loss_t1"].quantile(label_q)

y_train = (train["loss_t1"] > thr).astype(int).to_numpy()
y_test  = (test["loss_t1"] > thr).astype(int).to_numpy()

X_train = train[feat_cols].to_numpy()
X_test  = test[feat_cols].to_numpy()

print("thr =", float(thr))
print("train positives:", int(y_train.sum()), "/", len(y_train), f"({y_train.mean():.4f})")
print("test positives :", int(y_test.sum()), "/", len(y_test), f"({y_test.mean():.4f})")

thr = 0.012771045393253737
train positives: 81 / 804 (0.1007)
test positives : 8 / 201 (0.0398)


## 2) Train models + Evaluate (ROC-AUC + Recall@K)

### Models
We compare two simple baselines:

- **Logistic Regression (linear baseline)**  
  A scaled linear model that outputs a probability score $\hat p_t$.

- **Random Forest (non-linear baseline)**  
  A tree-ensemble that can capture non-linear patterns and feature interactions
  (e.g., volatility regime + lagged returns).

### Evaluation metrics
Because “extreme loss” events are rare, we avoid relying on accuracy.

- **ROC-AUC (ranking quality)**  
  Measures whether the model tends to assign higher scores to positives than negatives.
  If the test set contains only one class, ROC-AUC is not defined.

- **Alert-budget metric: Recall@K**  
  In practice, we often can only send **K alerts per day** (or per period).  
  We rank days by predicted score $\hat p_t$ and look at the top-K.

  $$ \mathrm{Recall@K} = \frac{\#\{\text{true positives among top-K}\}}{\#\{\text{all true positives}\}} $$

  Interpretation:
  - Recall@10 answers: “If we send 10 alerts, what fraction of extreme-loss days do we catch?”
  - Higher Recall@K is better for risk alerting (but may increase false alarms).

### Output to inspect
- ROC-AUC for Logit and RF
- Recall@K for K ∈ {5, 10, 20, 50}

In [22]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def recall_at_k(y_true, scores, k):
    total_pos = int(np.sum(y_true))
    if total_pos == 0:
        return np.nan
    k = min(k, len(scores))
    idx = np.argsort(scores)[::-1][:k]
    tp = int(np.sum(y_true[idx]))
    return tp / total_pos

# --- Logistic baseline ---
logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=4000, solver="lbfgs", class_weight="balanced"),
)
logit.fit(X_train, y_train)
proba_logit = logit.predict_proba(X_test)[:, 1]

print("[Logit] ROC-AUC:", roc_auc_score(y_test, proba_logit) if len(np.unique(y_test)) == 2 else "NA")

print("[Logit] Recall@K:")
for k in [5, 10, 20, 50]:
    print(f"  Recall@{k}: {recall_at_k(y_test, proba_logit, k):.4f}")

# --- RandomForest baseline ---
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=0,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("\n[RF] ROC-AUC:", roc_auc_score(y_test, proba_rf) if len(np.unique(y_test)) == 2 else "NA")

print("[RF] Recall@K:")
for k in [5, 10, 20, 50]:
    print(f"  Recall@{k}: {recall_at_k(y_test, proba_rf, k):.4f}")

[Logit] ROC-AUC: 0.4533678756476684
[Logit] Recall@K:
  Recall@5: 0.0000
  Recall@10: 0.0000
  Recall@20: 0.0000
  Recall@50: 0.2500

[RF] ROC-AUC: 0.694300518134715
[RF] Recall@K:
  Recall@5: 0.0000
  Recall@10: 0.1250
  Recall@20: 0.1250
  Recall@50: 0.5000


## 3) Summary

- If RF improves ROC-AUC and Recall@K, it suggests non-linear interactions among lagged returns and volatility/state features.
- Because positives are rare, top-K alert metrics are more practical than a single fixed threshold.

---